<a href="https://www.kaggle.com/code/henrykgreysson/competition?scriptVersionId=305023188" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [5]:

import torch, subprocess, sys
print("torch version:", torch.__version__)
print("torch built with CUDA:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("device count:", torch.cuda.device_count())
    print("device 0:", torch.cuda.get_device_name(0))
    print("allocated:", torch.cuda.memory_allocated()/1e9, "GB")
    print("reserved :", torch.cuda.memory_reserved()/1e9, "GB")
else:
    print("No CUDA runtime available in this session.")
    try:
        out = subprocess.check_output(["nvidia-smi"], text=True)
        print("\nnvidia-smi:\n", out[:500])
    except Exception as e:
        print("nvidia-smi not available:", e)

torch version: 2.9.0+cu126
torch built with CUDA: 12.6
cuda available: True
device count: 1
device 0: NVIDIA H100 80GB HBM3
allocated: 0.0 GB
reserved : 0.0 GB


In [ ]:
## !/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import time
import json
import hashlib
from collections import defaultdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, logging

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
logging.set_verbosity_error()

MODEL_DIR = "/kaggle/input/models/deepseek-ai/deepseek-math/pytorch/deepseek-math-7b-instruct/1"

INT_MIN, INT_MAX = -10**18, 10**18
FINAL_MIN, FINAL_MAX = 0, 99999

DEBUG_MODE = False

MAX_LLM_CALLS_PER_PROBLEM = 2
MAX_NEW_TOKENS_MAIN = 512
MAX_NEW_TOKENS_RETRY = 512
MAX_GEN_TIME_SECONDS = 14.0

GEN_REPETITION_PENALTY = 1.02

CACHE = {"gen": {}, "tool": {}}

MIN_LLM_SCORE_ACCEPT = 0.05


def dprint(*args, **kwargs):
    if DEBUG_MODE:
        print(*args, **kwargs)


def clamp_int(x):
    try:
        x = int(x)
        return max(INT_MIN, min(INT_MAX, x))
    except Exception:
        return None


def clamp_final(x):
    try:
        x = int(x)
        if FINAL_MIN <= x <= FINAL_MAX:
            return x
        return None
    except Exception:
        return None


def canonical_mod(x: int, m: int) -> int:
    return x % m if m and m > 0 else x


def _cache_key(*parts):
    s = "||".join(str(p) for p in parts)
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


def _cached_get(bucket: str, key: str):
    return CACHE.get(bucket, {}).get(key, None)


def _cached_set(bucket: str, key: str, value):
    CACHE.setdefault(bucket, {})[key] = value


EXPLICIT_ANSWER_RE = re.compile(
    r"(?im)^\s*(?:final\s*answer|answer|ans|result|output)\s*[:=\-→>]*\s*\(?\s*(-?\d{1,18})\s*\)?\s*\.?\s*$"
)
BOXED_RE = re.compile(r"\\boxed\s*\{\s*(-?\d{1,18})\s*\}")
GENERIC_INT_RE = re.compile(r"-?\d{1,18}")


def parse_answer_with_quality(text: str) -> Tuple[Optional[int], float, str]:
    if not text:
        return None, 0.0, "empty"

    ms = list(EXPLICIT_ANSWER_RE.finditer(text))
    if ms:
        v = clamp_int(ms[-1].group(1))
        if v is not None:
            return v, 1.0, "explicit_answer_tag"

    bx = list(BOXED_RE.finditer(text))
    if bx:
        v = clamp_int(bx[-1].group(1))
        if v is not None:
            return v, 0.75, "boxed_answer"

    tail = [ln.strip() for ln in text.splitlines() if ln.strip()][-4:]
    for ln in reversed(tail):
        m = re.fullmatch(r"\(?\s*(-?\d{1,18})\s*\)?\.?", ln)
        if m:
            v = clamp_int(m.group(1))
            if v is not None:
                return v, 0.45, "tail_integer_line"

    # Better fallback: prefer integers near the end
    lines = text.strip().splitlines()[-5:]
    nums = []
    for ln in lines:
        nums += GENERIC_INT_RE.findall(ln)
    
    if nums:
        v = clamp_int(nums[-1])
        if v is not None:
            return v, 0.25, "loose_tail_integer"


def latex_to_plain(s: str) -> str:
    if not s:
        return s
    t = s
    t = t.replace("−", "-").replace("—", "-").replace("–", "-").replace("⋅", "*").replace("×", "*")
    t = re.sub(r"\\left\s*", "", t)
    t = re.sub(r"\\right\s*", "", t)
    t = t.replace(r"\mathbb{Z}", "Z").replace(r"\mathbb{N}", "N").replace(r"\mathbb{Q}", "Q").replace(r"\mathbb{R}", "R")
    t = t.replace(r"\geq", ">=").replace(r"\ge", ">=").replace(r"\leq", "<=").replace(r"\le", "<=")
    t = t.replace(r"\cdot", "*").replace(r"\times", "*")
    t = t.replace(r"\equiv", "≡")
    t = re.sub(r"\\pmod\s*\{([^{}]+)\}", r"(mod \1)", t)
    t = re.sub(r"\\bmod\s*\{?\s*([^){}\s]+)\s*\}?", r"(mod \1)", t)
    t = re.sub(r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"(\1)/(\2)", t)
    t = t.replace("$$", " ").replace("$", " ").replace(r"\(", " ").replace(r"\)", " ").replace(r"\[", " ").replace(r"\]", " ")
    t = re.sub(r"\s+", " ", t).strip()
    return t


def preprocess_problem(problem: str) -> str:
    return latex_to_plain(problem or "")


def egcd(a, b):
    if b == 0:
        return a, 1, 0
    g, x, y = egcd(b, a % b)
    return g, y, x - (a // b) * y


def modinv(a, m):
    a %= m
    g, x, _ = egcd(a, m)
    return x % m if g == 1 else None


def factorize(n: int):
    n = abs(int(n))
    f = {}
    if n < 2:
        return f
    while n % 2 == 0:
        f[2] = f.get(2, 0) + 1
        n //= 2
    p = 3
    while p * p <= n:
        while n % p == 0:
            f[p] = f.get(p, 0) + 1
            n //= p
        p += 2
    if n > 1:
        f[n] = f.get(n, 0) + 1
    return f


def tau(n: int):
    if n == 0:
        return 0
    ans = 1
    for e in factorize(n).values():
        ans *= (e + 1)
    return ans


def vp_fact(n: int, p: int):
    s = 0
    while n:
        n //= p
        s += n
    return s


def s_basic_arith(problem: str):
    m = re.fullmatch(r"compute\s+(-?\d+)\s*\+\s*(-?\d+)\.?", problem.lower().strip())
    return int(m.group(1)) + int(m.group(2)) if m else None


def s_mod_inverse(problem: str):
    m = re.search(r"least positive integer n such that\s*(-?\d+)n\s*≡\s*1\s*\(mod\s*(-?\d+)\)", problem.lower())
    if not m:
        return None
    a, mod = int(m.group(1)), int(m.group(2))
    return modinv(a, mod) if mod > 0 else None


def s_divisor_count(problem: str):
    m = re.search(r"how many positive divisors does\s*(\d+)\s*have", problem.lower())
    return tau(int(m.group(1))) if m else None


def s_trailing_zeros(problem: str):
    m = re.search(r"how many trailing zeros are in\s*(\d+)!\s*\??", problem.lower())
    return vp_fact(int(m.group(1)), 5) if m else None


GENERIC_SOLVERS = [
    (s_basic_arith, 0.99),
    (s_mod_inverse, 0.98),
    (s_divisor_count, 0.98),
    (s_trailing_zeros, 0.99),
]


def deterministic_candidates(problem: str):
    p2 = preprocess_problem(problem)
    out = []
    for fn, conf in GENERIC_SOLVERS:
        try:
            v = fn(p2)
            if v is not None:
                out.append((clamp_int(v), conf, fn.__name__))
        except Exception:
            pass
    return out


def _extract_required_remainder_mod(problem: str):
    p = preprocess_problem(problem).lower()
    m = re.search(r"remainder.*divided by\s+(\d+)", p)
    if m:
        return int(m.group(1))
    m2 = re.search(r"mod\s+(\d+)", p)
    if m2:
        return int(m2.group(1))
    return None


def _problem_features(problem: str):
    p = preprocess_problem(problem).lower()
    return {
        "has_count": ("how many" in p or "number of" in p or "count" in p),
        "has_remainder": ("remainder" in p or "mod" in p),
    }


def _is_hard_reasoning_problem(problem: str) -> bool:
    p = preprocess_problem(problem).lower()
    hard_markers = [
        "triangle", "circumcircle", "incircle", "tournament", "largest possible",
        "across all", "for all positive integers", "unique such", "floor",
        "remainder when", "divided by 10^5", "imo", "aime", "function"
    ]
    return any(k in p for k in hard_markers) or len(p) > 220


def verify_candidate(problem: str, candidate: int, parse_quality: float = 1.0):
    if candidate is None:
        return 0.0, {"ok": False, "reason": "none_candidate"}

    c = clamp_int(candidate)
    if c is None:
        return 0.0, {"ok": False, "reason": "non_int"}

    feats = _problem_features(problem)
    q = _extract_required_remainder_mod(problem)

    score = 1.0
    score *= max(0.05, float(parse_quality))

    if feats["has_remainder"] and q is not None and q > 0:
        if not (0 <= c < q):
            return 0.0, {"ok": False, "reason": "remainder_out_of_range"}

    if feats["has_count"] and c < 0:
        return 0.0, {"ok": False, "reason": "negative_count"}

    if not (INT_MIN <= c <= INT_MAX):
        return 0.0, {"ok": False, "reason": "int_out_of_bounds"}

    return max(0.0, min(1.0, score)), {"ok": True}


print("Loading model1:", MODEL_DIR)
tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, use_fast=False, trust_remote_code=True)
if tok.pad_token is None and tok.eos_token is not None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto",
).eval()


@torch.inference_mode()
def _gen(msgs, max_new_tokens=512):
    chat = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    x = tok(chat, return_tensors="pt")
    try:
        dev = next(model.parameters()).device
        x = {k: v.to(dev) for k, v in x.items()}
    except Exception:
        pass
    y = model.generate(
        **x,
        max_new_tokens=max_new_tokens,
        max_time=MAX_GEN_TIME_SECONDS,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
        use_cache=True,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=GEN_REPETITION_PENALTY,
    )
    return tok.decode(y[0, x["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def _cached_gen(msgs, max_new_tokens=512, tag=""):
    k = _cache_key(tag, json.dumps(msgs, ensure_ascii=False, sort_keys=True), max_new_tokens)
    hit = _cached_get("gen", k)
    if hit is not None:
        return hit
    out = _gen(msgs, max_new_tokens=max_new_tokens)
    _cached_set("gen", k, out)
    return out


MAIN_SYSTEM_PROMPT = (
    "Solve the problem carefully.\n"
    "You MUST end with EXACTLY this format:\n"
    "Final Answer: <integer>\n"
    "No extra text after."
)

RETRY_SYSTEM_PROMPT = (
    "Solve again carefully.\n"
    "Output ONLY:\n"
    "Final Answer: <integer>"
)


def llm_solve_once(problem: str):
    p2 = preprocess_problem(problem)

    msgs = [
        {"role": "system", "content": MAIN_SYSTEM_PROMPT},
        {"role": "user", "content": p2},
    ]

    answers = []

    for i in range(5):  # 5 samples
        raw = _gen(msgs, max_new_tokens=MAX_NEW_TOKENS_MAIN)
        print(f"\n=== SAMPLE {i} ===\n", raw)

        v, pq, mode = parse_answer_with_quality(raw)
        if v is not None:
            answers.append((v, pq, raw))

    if not answers:
        return None, "", 0.0, "no_parse"

    # vote
    counter = defaultdict(int)
    for v, _, _ in answers:
        counter[v] += 1

    best = max(counter.items(), key=lambda x: x[1])[0]

    # pick best-quality version of that answer
    best_entry = max([a for a in answers if a[0] == best], key=lambda x: x[1])

    return best_entry[0], best_entry[2], best_entry[1], "multi_sample"


def llm_retry(problem: str, prev_raw: str):
    p2 = preprocess_problem(problem)
    msgs = [
        {"role": "system", "content": RETRY_SYSTEM_PROMPT},
        {"role": "user", "content": f"Problem:\n{p2}\n\nPrevious answer:\n{prev_raw}"},
    ]
    raw = _cached_gen(msgs, max_new_tokens=MAX_NEW_TOKENS_RETRY, tag="retry_v5_fast")
    v, pq, mode = parse_answer_with_quality(raw)
    return v, raw, pq, mode


def _safe_small_bruteforce(problem: str):
    p = preprocess_problem(problem).lower().strip()
    m = re.fullmatch(r"compute\s+(-?\d+)\s*\+\s*(-?\d+)\.?", p)
    if m:
        return int(m.group(1)) + int(m.group(2))
    m = re.fullmatch(r"compute\s+(-?\d+)\s*-\s*(-?\d+)\.?", p)
    if m:
        return int(m.group(1)) - int(m.group(2))
    m = re.fullmatch(r"compute\s+(-?\d+)\s*\*\s*(-?\d+)\.?", p)
    if m:
        return int(m.group(1)) * int(m.group(2))
    return None


def solve_one(problem: str) -> Dict[str, Any]:
    cands = []
    llm_calls = 0

    for v, conf, src in deterministic_candidates(problem):
        if v is None:
            continue
        s, _ = verify_candidate(problem, v, parse_quality=1.0)
        if s > 0:
            cands.append({"val": v, "score": conf * s, "src": f"det:{src}"})

    bf = _safe_small_bruteforce(problem)
    if bf is not None:
        v = clamp_int(bf)
        s, _ = verify_candidate(problem, v, parse_quality=1.0)
        if s > 0:
            cands.append({"val": v, "score": 0.995 * s, "src": "det:small_bruteforce"})

    raw1 = ""
    v1 = None
    pq1 = 0.0
    if llm_calls < MAX_LLM_CALLS_PER_PROBLEM:
        v1, raw1, pq1, _ = llm_solve_once(problem)
        llm_calls += 1
        if v1 is not None:
            s1, _ = verify_candidate(problem, v1, parse_quality=pq1)
            if s1 >= MIN_LLM_SCORE_ACCEPT:
                cands.append({"val": v1, "score": 0.90 * s1, "src": f"llm:main(pq={pq1:.2f})"})

    need_retry = False
    if raw1 == "":
        need_retry = True
    elif v1 is None:
        need_retry = True
    elif pq1 < 0.70:
        need_retry = True

    if need_retry and llm_calls < MAX_LLM_CALLS_PER_PROBLEM:
        v2, raw2, pq2, _ = llm_retry(problem, raw1)
        llm_calls += 1
        if v2 is not None:
            s2, _ = verify_candidate(problem, v2, parse_quality=pq2)
            if s2 >= MIN_LLM_SCORE_ACCEPT:
                cands.append({"val": v2, "score": 0.86 * s2, "src": f"llm:retry(pq={pq2:.2f})"})

    if not cands:
        return {
            "pred": 0,
            "confidence": 0.0,
            "method": "strict_reject_no_valid_candidate",
            "debug": {"llm_calls": llm_calls}
        }

    agg = defaultdict(float)
    for c in cands:
        agg[c["val"]] += max(0.0, c["score"])

    ranked = sorted(agg.items(), key=lambda kv: kv[1], reverse=True)
    best_raw, best_w = ranked[0]
    second_w = ranked[1][1] if len(ranked) > 1 else 1e-9

    q = _extract_required_remainder_mod(problem)
    best = canonical_mod(best_raw, q) if (q is not None and q > 0) else best_raw

    feats = _problem_features(problem)
    if feats["has_count"] and best < 0:
        return {
            "pred": 0,
            "confidence": 0.0,
            "method": "strict_reject_negative_count",
            "debug": {"llm_calls": llm_calls, "ranked_top": ranked[:5]}
        }

    best = clamp_final(best)
    if best is None:
        return {
            "pred": 0,
            "confidence": 0.0,
            "method": "strict_reject_out_of_range",
            "debug": {"llm_calls": llm_calls, "ranked_top": ranked[:5]}
        }

    # confidence with parse-quality-aware cap
    contributors = [c for c in cands if c["val"] == best_raw]
    source_top = max(contributors, key=lambda z: z["score"]) if contributors else {"src": ""}
    src = source_top["src"]

    total_w = sum(w for _, w in ranked) + 1e-9
    rel_conf = best_w / total_w
    margin = best_w / max(1e-9, second_w)

    base_conf = 0.55 * rel_conf + 0.25 * min(1.0, margin / 2.0)

    pq_cap = 0.60
    m = re.search(r"pq=([0-9.]+)", src)
    if m:
        pq = float(m.group(1))
        if pq >= 0.95:
            pq_cap = 0.92
        elif pq >= 0.75:
            pq_cap = 0.80
        elif pq >= 0.45:
            pq_cap = 0.68

    confidence = max(0.0, min(pq_cap, base_conf + 0.15))

    return {
        "pred": int(best),
        "confidence": float(confidence),
        "method": "single_model_relaxed_parse_fast_v2",
        "debug": {
            "llm_calls": llm_calls,
            "ranked_top": ranked[:5],
            "num_candidates": len(cands),
            "top_ratio": float(margin),
        }
    }


def solve_batch(problem_list: List[str], expected: Optional[List[int]] = None):
    rows = []
    t0 = time.time()

    for i, q in enumerate(problem_list, 1):
        out = solve_one(q)
        row = {"idx": i, "pred": out["pred"], "method": out["method"], "conf": out["confidence"]}
        if expected is not None and i <= len(expected):
            row["expected"] = expected[i - 1]
            row["correct"] = (out["pred"] == expected[i - 1])
        rows.append(row)
        print(f"[{i}] pred={out['pred']} method={out['method']} conf={out['confidence']:.3f}")

    elapsed = time.time() - t0
    print(f"elapsed={elapsed:.2f}s")
    df = pd.DataFrame(rows)
    if expected is not None and "correct" in df.columns:
        print(f"Accuracy: {df['correct'].sum()}/{len(df)} = {df['correct'].mean():.1%}")
    return df


def solve_batch_dict(samples: List[Dict[str, Any]]):
    rows = []
    t0 = time.time()
    ok = 0
    scored_count = 0

    for i, s in enumerate(samples, 1):
        q = s["q"] if isinstance(s, dict) and "q" in s else str(s)
        out = solve_one(q)
        pred = int(out["pred"])

        row = {"idx": i, "pred": pred, "method": out["method"], "conf": out["confidence"]}

        if isinstance(s, dict) and ("expected" in s):
            exp = int(s["expected"])
            corr = (pred == exp)
            ok += int(corr)
            scored_count += 1
            row["expected"] = exp
            row["correct"] = corr
            print(f"[{i}] pred={pred} exp={exp} {'✅' if corr else '❌'} | {out['method']} | conf={out['confidence']:.3f}")
        else:
            print(f"[{i}] pred={pred} | {out['method']} | conf={out['confidence']:.3f}")

        rows.append(row)

    elapsed = time.time() - t0
    df = pd.DataFrame(rows)
    if scored_count > 0:
        print(f"\nAccuracy: {ok}/{scored_count} = {ok/max(1, scored_count):.1%} | elapsed={elapsed:.2f}s")
    else:
        print(f"\nElapsed={elapsed:.2f}s")
    return df


if __name__ == "__main__":
    samples = [
        {"q":"Alice and Bob are each holding some integer number of sweets. Alice says to Bob: “If we each added the number of sweets we’re holding to our (positive integer) age, my answer would be double yours. If we took the product, then my answer would be four times yours.” Bob replies: “Why don’t you give me five of your sweets because then both our sum and product would be equal.” What is the product of Alice and Bob’s ages?","expected":50},
        {"q":"A 500 × 500 square is divided into k rectangles, each having integer side lengths. Given that no two of these rectangles have the same perimeter, the largest possible value of k is K. What is the remainder when K is divided by 10^5?","expected":520},
        {"q":"Let ABC be an acute-angled triangle with integer side lengths and AB < AC. Points D and E lie on segments BC and AC, respectively, such that AD = AE = AB. Line DE intersects AB at X. Circles BXD and CED intersect for the second time at Y ≠ D. Suppose that Y lies on line AD. There is a unique such triangle with minimal perimeter. This triangle has side lengths a = BC, b = CA, and c = AB. Find the remainder when abc is divided by 10^5.","expected":336},
        {"q":"A tournament is held with 2^20 runners each of which has a different running speed. In each race, two runners compete against each other with the faster runner always winning the race. The competition consists of 20 rounds with each runner starting with a score of 0. In each round, the runners are paired in such a way that in each pair, both runners have the same score at the beginning of the round. The winner of each race in the i-th round receives 2^(20−i) points and the loser gets no points. At the end of the tournament, we rank the competitors according to their scores. Let N denote the number of possible orderings of the competitors at the end of the tournament. Let k be the largest positive integer such that 10^k divides N. What is the remainder when k is divided by 10^5?","expected":21818},
        {"q":"Let f: Z_{≥1} → Z_{≥1} be a function such that for all positive integers m and n, f(m) + f(n) = f(m + n + mn). Across all functions f such that f(n) ≤ 1000 for all n ≤ 1000, how many different values can f(2024) take?","expected":580},
        {"q":"Define a function f: Z_{≥1} → Z_{≥1} by f(n) = sum_{i=1..n} sum_{j=1..n} j^1024 * floor( 1/j + (n-i)/n ). Let M = 2·3·5·7·11·13 and let N = f(M^15) − f(M^15 − 1). Let k be the largest non-negative integer such that 2^k divides N. What is the remainder when 2^k is divided by 5^7?","expected":32951},
        {"q":"Let ABC be a triangle with AB ≠ AC, circumcircle Ω, and incircle ω. Let the contact points of ω with BC, CA, and AB be D, E, and F, respectively. Let the circumcircle of AFE meet Ω at K and let the reflection of K in EF be K′. Let N denote the foot of the perpendicular from D to EF. The circle tangent to line BN and passing through B and K intersects BC again at T ≠ B. Let sequence (F_n) be defined by F_0 = 0, F_1 = 1 and for n ≥ 2, F_n = F_{n−1} + F_{n−2}. Call ABC n-tastic if BD = F_n, CD = F_{n+1}, and K N K′ B is cyclic. Across all n-tastic triangles, let a_n denote the maximum possible value of (CT·NB)/(BT·NE). Let α denote the smallest real number such that for all sufficiently large n, a_{2n} < α^n. Given that α = p + √q for rationals p and q, what is the remainder when floor(p^q) is divided by 99991?","expected":57447},
        {"q":"On a blackboard, Ken starts off by writing a positive integer n and then applies the following move until he first reaches 1. Given that the number on the board is m, he chooses a base b, where 2 ≤ b ≤ m, and considers the unique base-b representation of m, m = sum_{k=0..∞} a_k b^k, where a_k are non-negative integers and 0 ≤ a_k < b for each k. Ken then erases m on the blackboard and replaces it with sum_{k=0..∞} a_k. Across all choices of 1 ≤ n ≤ 10^10, the largest possible number of moves Ken could make is M. What is the remainder when M is divided by 10^5?","expected":32193},
        {"q":"Let F be the set of functions α: Z → Z for which there are only finitely many n ∈ Z such that α(n) ≠ 0. For two functions α and β in F, define their product α ⋆ β to be sum_{n∈Z} α(n)β(n). Also, for n ∈ Z, define a shift operator S_n: F → F by S_n(α)(t) = α(t + n) for all t ∈ Z. A function α ∈ F is called shiftfy if (i) α(m) = 0 for all integers m < 0 and m > 8, and (ii) There exists β ∈ F and integers k ≠ l such that for all n ∈ Z, S_n(α) ⋆ β = 1 if n ∈ {k,l}, and 0 if n ∉ {k,l}. How many shiftfy functions are there in F?","expected":160},
        {"q":"Let n ≥ 6 be a positive integer. We call a positive integer n-Norwegian if it has three distinct positive divisors whose sum is equal to n. Let f(n) denote the smallest n-Norwegian positive integer. Let M = 3^2025!, and for a non-negative integer c define g(c) = (1/2025!) * floor( 2025! * f(M + c) / M ). We can write g(0) + g(4M) + g(1848374) + g(10162574) + g(265710644) + g(44636594) = p/q where p and q are coprime positive integers. What is the remainder when p + q is divided by 99991?","expected":8687},
    ]
    df = solve_batch_dict(samples)
    print(df)